In [1]:
import geopandas as gpd
import numpy as np
import math
from scipy.spatial import Voronoi,voronoi_plot_2d
import random as rnd
from IPython.display import display, HTML,Javascript,clear_output
from svg_plot import SVGPlot
import time
import csv


In [2]:
areas = []
population=[]

In [3]:
def point_in_polygon(x, y, polygon):
    polygon = np.asarray(polygon)
    n = len(polygon)
    inside = False
    
    for i in range(n):
        x1, y1 = polygon[i]
        x2, y2 = polygon[(i + 1) % n]
        
        # Check if edge crosses horizontal ray to the right
        if ((y1 > y) != (y2 > y)) and (x < (x2 - x1) * (y - y1) / (y2 - y1) + x1):
            inside = not inside
    
    return inside

In [4]:
class PopulationArea:
    def __init__(self,area_id,admin_area,area_type,vertices):
        self.area_id = area_id
        self.admin_area = admin_area
        self.area_type = area_type
        self.vertices = vertices
        self.local_population = []

In [5]:
def get_area(x,y,areas):
    for area in areas.values():
        if point_in_polygon(x,y,area.vertices):
            return True, area
    return False, None
    

In [6]:
def linear_toward_zero():
    u = rnd.random()
    return 1 - math.sqrt(1 - u)

In [7]:
def age_migrate_curve(x, start_point=20, end_point=70, decay_rate=None):
    if x <20 :
        return 0.0
    x = np.array(x) if hasattr(x, '__iter__') else np.array([x])
    if decay_rate is None:
        decay_rate = -np.log(0.01) / (end_point - start_point)
    shifted_x = x - start_point
    prob = np.exp(-decay_rate * shifted_x)
    prob = np.clip(prob, 0, 1)
    if len(prob) == 1:
        return prob[0]
    return prob

In [8]:
def generate_age():
    def age_probability(age):
        if age < 0:
            return 0
        
        # Peak at ~38, rise then fall
        peak = 38
        rise = (age / peak) ** 1.5
        fall = np.exp(-0.015 * (age - peak))
        
        return rise * fall
    age = -1
    while age < 0:
        candidate = np.random.uniform(0, 100)
        p = age_probability(candidate)
        max_p = age_probability(38)  # Peak probability
        
        if np.random.rand() < (p / max_p):
            return int(candidate)


In [9]:
class Denizen:
    def __init__(self, area):
        self.area = area
        self.pos = (0, 0)
        self.age = generate_age()
        self.moved = False

    def assign_pos(self,):
        # FIXED: Extract correct coordinates
        x_vals = [point[0] for point in self.area.vertices]  # x is first element
        y_vals = [point[1] for point in self.area.vertices]  # y is second element
        
        x_min, x_max = min(x_vals), max(x_vals)
        y_min, y_max = min(y_vals), max(y_vals)
        
        while True:
            # FIXED: Correct random range
            x = x_min + rnd.random() * (x_max - x_min)
            y = y_min + rnd.random() * (y_max - y_min)
            
            if point_in_polygon(x, y, self.area.vertices):
                self.pos = (float(x), float(y))
                self.area.local_population.append(self)
                return

    def migrate(self):
        if rnd.random()<age_migrate_curve(self.age):
            return True


    def __str__(self):
        return f"Vector({self.area.area_id}, {self.pos},{self.age})"

In [10]:
def migrate():
    for p in population:
        if (p.moved)==False and (p.migrate()):
            flag = False
            while flag == False:
                x = linear_toward_zero()*10
                y = linear_toward_zero()*10
                flag, area = get_area(x,y,areas)
                if (flag == True) and (area.area_id !=7):
                    p.pos=(x,y)
                    p.area.local_population.remove(p)
                    p.area= area
                    area.local_population.append(p)
                    p.moved = True                

In [11]:
def get_areas(file_path, scale_factor=1.0):
    gdf = gpd.read_file(file_path)
    areas={}
    for row in gdf.itertuples(index=False):
        geom = row.geometry
        if geom is None:
            continue
        
        # Extract Coords
        if geom.geom_type == 'Polygon':
            coords = [
                [pt[0] * scale_factor, pt[1] * scale_factor] 
                for pt in geom.exterior.coords
            ]
        else:
            continue
            
        # Extract Properties (convert namedtuple to dict)
        # We skip the 'geometry' field which is the first element in itertuples
        props = {k: v for k, v in row._asdict().items() if k != 'geometry'}
        area = PopulationArea(props['id'],props['admin_area'],props['type'],coords)
        areas[props['id']]=area
    return areas

In [12]:
svg = SVGPlot(500, 500, 12)
areas =  get_areas('Meropis/bounded_polygons_small.geojson',10)
with open('Meropis/artifical_population.csv', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    
    for row in reader:
        # Each row is a dict with header names as keys
        id = row['area']  # Access by header name
        person = Denizen(areas[id])
        person.assign_pos()
        population.append(person)


output_display = display(HTML("<div id='sim-container'>Loading...</div>"), display_id=True)
for t in range (100):
    svg.clear()
    #for area in areas:
    for area in areas.values():
        svg.add_polygon(area.vertices,'rgb(220,220,220)')
    for i in range(len(population)):
        p = population[i]
        if p.moved:
            x = p.pos[0]
            y = p.pos[1]
            svg.add_rect(x,y,f'rgb(522,{p.age*3},{p.age*3})',0.04)
    migrate()   
    new_svg = svg.get_canvas()
    new_svg += f'<p> Iteration: {t} weeks </p>'
    output_display.update(HTML(f"<div id='sim-container'>{new_svg}</div>"))
    time.sleep(0.0)


KeyError: 'd12'